# 41. Random-label fitting on the indicator models

Random-label fitting asks a different question from the direct probe. The probe asks
whether the model can still learn something useful. This asks whether it can still change
at all: it gives the model 2,000 images with labels drawn at random and measures how far
the training loss falls. Lower loss means more capacity to change.

It was measured once before, on 20 and 21 October, by notebook 29. That run used the study
`paired_generalization_v2`. Everything measured on 1 to 3 November, the five indicators,
the matched probe and the random-direction control, used `indicator_stream_v1`, a separate
training run with different model files. So the one measurement that agrees with the probe
is the one measurement not read from the same models as the rest.

This notebook closes that gap. Same method, same task, same labels, same step budget, same
rate grid. The study changes, the arms are cut to `full`, the natural-task probe is dropped
because notebooks 38 and 40 already cover it, and the later probe point gets five
repetitions instead of two, which is where the October run's noise floor was widest.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")
if IN_COLAB:
    REPO = Path("/content/path-probe")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.probe import (RandomLabels, best_of_grid, noise_floor, probe_once,
                         relative_contribution, snapshot, unchanged)

sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError as problem:
    print('Progress panel unavailable, using plain bars:', problem)

    class Dashboard:
        """Just enough of the panel to keep the notebook running."""

        def __init__(self, *args, **kwargs): pass
        def __enter__(self): return self
        def __exit__(self, *args): return False
        def note(self, text): print(text, flush=True)

        def bar(self, total, description, unit='step'):
            from uncle.telemetry import progress_bar
            return progress_bar(total, description)

import torch
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)

## 2. Settings

Everything that decides what the number means is held at notebook 29's value: task 8, the
label seed, 2,000 images, 600 steps, the same three rates, the same lowest-training-loss
rule for picking among them.

Three things change on purpose, and each one is marked below.

In [ ]:
# The study the indicators, the matched probe and the control were all read
# from. This is the one change the notebook exists to make.
STUDY_ID = "indicator_stream_v1"
BUCKET = "uncle-experiments"
ORDER = "order_01"
SEEDS = [0, 1, 2]

# A name of its own, so the October results in partial.jsonl cannot be
# overwritten or mixed in. Delete this file to force a clean rerun.
RESUME_KEY = f"uncle/plasticity_probe/{STUDY_ID}/{ORDER}/matched1.jsonl"
SAVE_EVERY = 10               # upload after this many new runs

# CHANGE 2. Empty, so no natural-task probe runs. Notebook 38 measured the
# ends of this stream and notebook 40 measured five points along it, both on
# these same models, so repeating it here would buy nothing and cost two
# thirds of the running time.
PROBE_TASKS = []

# Held exactly at notebook 29's values. Task 8 is outside order 01, so the
# stream never saw it. The label seed fixes which wrong label each image gets,
# and both probe points share that one draw: a fresh draw per point would make
# the two points incomparable.
RANDOM_LABEL_TASK = "8"
RANDOM_LABEL_SEED = 1020
RANDOM_LABEL_IMAGES = 2000
RANDOM_LABEL_STEPS = 600      # about 19 passes over 2,000 images

# The same grid. A model late in a stream may need a different rate than a
# fresh one, so the probe tries all three and keeps the best. In October
# 1e-3 was never the one chosen, at either point; it is kept anyway so this
# run and that one choose from the same set.
RATES = [1e-4, 3e-4, 1e-3]

STEPS = 250                   # unused here, since PROBE_TASKS is empty
BATCH_SIZE = 64
FEATURE_IMAGES = 2000         # unused here: effective rank is an E2 measurement

# CHANGE 3. Notebook 29 used 5 and 2. The 2 is why seed 1's noise floor came
# out as wide as 0.42: the floor is built from the spread at t = 0 but is
# scaled by the count at both points. Five at each end costs 27 extra runs
# and narrows the floor.
REPETITIONS_AT_ZERO = 5
REPETITIONS_LATER = 5

# CHANGE 4. The full arm only. The other two arms answered a different
# question, whether the 32-number task code earns its place, and that was
# settled on 21 October. Keeping them would triple the cost.
ARMS = [
    ("full", False, True),   # generator and code both train
]

SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"

# One random-label job per point, so the task count is 1 whatever PROBE_TASKS holds.
runs_at = lambda reps: (len(PROBE_TASKS) + 1) * len(RATES) * reps * len(ARMS)
TOTAL_RUNS = (runs_at(REPETITIONS_AT_ZERO) + runs_at(REPETITIONS_LATER)) * len(SEEDS)
# Recorded with every run. October's saved runs have no GPU name, so if this
# run and that one ever get compared directly, this is the field that says
# whether the machines matched.
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", GPU_NAME)
print(f"Probe runs: {TOTAL_RUNS}  "
      f"({len(RATES)} rates, {len(ARMS)} arm, "
      f"{REPETITIONS_AT_ZERO} and {REPETITIONS_LATER} repetitions, "
      f"{len(SEEDS)} seeds)")
# 1.34 minutes is the measured median of October's 63 random-label runs, not a guess.
print(f"At the 1.34 minutes October's runs took, that is about "
      f"{TOTAL_RUNS * 1.34 / 60:.1f} hours, plus model downloads.")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the images and the two models

For each seed, `t = 8` is that seed's finished source model and `t = 0` is a generator
built with the same settings and the same seed, which has learned nothing. The models are
built one seed at a time, so only one is ever in memory.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
CLASSES = plan["config"]["classes_per_task"]

overlap = set(PROBE_TASKS + [RANDOM_LABEL_TASK]) & set(ORDER_TASKS)
if overlap:
    raise ValueError(f"These probe tasks are in the learning order: {sorted(overlap)}")

needed = tuple(ORDER_TASKS + PROBE_TASKS + [RANDOM_LABEL_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded images for", len(needed), "tasks")

def source_state(seed):
    """That seed's finished eight-task model, read from R2."""
    job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{seed}")
    state = storage.load(job)
    if state is None or state["report"]["status"] != "complete":
        raise RuntimeError(f"Source model for seed {seed} is not finished in R2.")
    return state

# Check every seed before any training starts, so a missing source model stops
# the notebook now rather than two hours into the run.
for seed in SEEDS:
    source_state(seed)
print("Source models present for seeds:", SEEDS)

def build(seed, restore_source):
    """A model at one probe point. restore_source False gives the t = 0 model."""
    config = Config(**{**plan["config"], "seed": seed, "device": DEVICE,
                       "requests": tuple(("learn", t) for t in ORDER_TASKS)})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if restore_source:
        checkpoint.restore(source_state(seed), hypernet=hypernet, uncle=trainer)
    return target, hypernet, trainer

# The random labels are drawn once and shared by every probe point. A fresh
# draw per point would make the points incomparable.
from torch.utils.data import Subset
# Fewer images, so the fixed step budget gives enough passes to memorise them.
# The subset is drawn once with a fixed seed and shared by every probe point.
pool = tasks[RANDOM_LABEL_TASK]["train"]
chosen = torch.randperm(len(pool),
                        generator=torch.Generator().manual_seed(RANDOM_LABEL_SEED)
                        )[:RANDOM_LABEL_IMAGES].tolist()
random_label_set = RandomLabels(Subset(pool, chosen), CLASSES, RANDOM_LABEL_SEED)
print(f"Random-label probe: {len(random_label_set)} images, {RANDOM_LABEL_STEPS} steps, "
      f"about {RANDOM_LABEL_STEPS * BATCH_SIZE / len(random_label_set):.0f} passes")
print("Probe tasks:", PROBE_TASKS, "| random labels from task", RANDOM_LABEL_TASK)

## 5. Run the grid

Every probe run snapshots the model, trains a fresh code on a probe task, measures, then
puts the model back. The cell checks after each run that the model really is unchanged, and
stops if it is not.

Each run uses one of three arms. The arm decides which parts are allowed to train:

| Arm | Generator | 32-number task code | What it tells you |
| --- | --- | --- | --- |
| full | trains | trains | how well the model learns a new task |
| code-only | frozen | trains | whether the code alone can learn a task |
| generator-only | trains | frozen at its random start | whether training the code buys anything |

The first two are the adapter study's own arms. The third is the mirror of the second and
answers a separate open question, at the cost of half again as much time.

**This cell can be stopped and started again.** Finished runs are written to a file in R2
every few runs. Running the cell again reads that file back, skips what is already done,
and continues. A session that disconnects after two hours therefore costs nothing.

In [ ]:
import time

# ---- resume -----------------------------------------------------------------
# A run is identified by everything that makes it different from another run.
# Two runs sharing an identity would be measuring the same thing twice.
def identity(run):
    return (run["seed_label"], run["point"], run["endpoint"], run["task"],
            run["learning_rate"], run["arm"], run["repetition"])

PARTIAL = CACHE / "diagnostics" / "plasticity_probe_partial.jsonl"
PARTIAL.parent.mkdir(parents=True, exist_ok=True)

results = []
if store.info(RESUME_KEY) is not None:
    # One JSON object per line, so a half-written final line can be dropped
    # without losing everything before it.
    saved = store.download(RESUME_KEY, CACHE / "cache")
    for line in saved.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    # An earlier version of this notebook compared arm names that were spelled
    # two different ways, so two of the three arms were never skipped and were
    # run again every session. Keep the first copy of each run and drop the
    # rest, otherwise the best-of-grid pick below chooses from more samples for
    # some arms than for others, which biases them.
    first, deduplicated = {}, []
    for run in results:
        key = identity(run)
        if key not in first:
            first[key] = True
            deduplicated.append(run)
    if len(deduplicated) < len(results):
        print(f"Dropped {len(results) - len(deduplicated)} repeated runs "
              f"left by the earlier arm-name problem.")
        results = deduplicated
    print(f"Resuming: {len(results)} runs already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")
done = {identity(r) for r in results}

def save_partial():
    store.upload(PARTIAL, RESUME_KEY, overwrite=True)

# ---- the grid ---------------------------------------------------------------
new_runs = 0

def grid_at(seed, point, restore_source, repetitions, board):
    """Every probe task, rate, arm and repetition at one probe point."""
    global new_runs
    jobs_here = [(task, tasks[task]["train"], tasks[task]["test"], STEPS, "E2")
                 for task in PROBE_TASKS]
    jobs_here.append((RANDOM_LABEL_TASK, random_label_set, None,
                      RANDOM_LABEL_STEPS, "E1"))

    # Work out what is left before building anything, so a probe point that is
    # already finished costs no model loading at all.
    todo = [(task, train_set, test_set, steps, endpoint, rate, arm, code_only,
             train_code, repetition)
            for task, train_set, test_set, steps, endpoint in jobs_here
            for rate in RATES
            for arm, code_only, train_code in ARMS
            for repetition in range(repetitions)
            if (seed, point, endpoint, task, rate, arm, repetition) not in done]
    if not todo:
        board.note(f"seed {seed}, t = {point}: already finished, skipping")
        return

    target, hypernet, trainer = build(seed, restore_source)
    reference = snapshot(hypernet, trainer)
    bar = board.bar(len(todo), f"seed {seed} t={point}", unit="run")
    for (task, train_set, test_set, steps, endpoint, rate, arm, code_only,
         train_code, repetition) in todo:
        board.note(f"seed {seed} {endpoint} task {task} lr {rate:g} "
                   f"{arm} rep {repetition + 1}")
        started = time.perf_counter()
        run = probe_once(
            hypernet, target, trainer, task=task,
            train_set=train_set, test_set=test_set, steps=steps,
            learning_rate=rate, batch_size=BATCH_SIZE,
            code_only=code_only, train_code=train_code,
            # This seed fixes the probe itself. It depends on the probe task
            # and the repetition, never on the probe point or the run's seed,
            # so two points differ only by the model's state.
            seed=12345 + 1000 * repetition + int(task),
            feature_images=FEATURE_IMAGES if endpoint == "E2" else 0)
        if not unchanged(hypernet, reference):
            raise RuntimeError("A probe changed the model. Stop and fix "
                               "this before trusting anything.")
        run.update(point=point, endpoint=endpoint, repetition=repetition,
                   seed_label=seed, minutes=(time.perf_counter() - started) / 60,
                   # Not part of identity(), so adding it does not invalidate
                   # the runs already saved in R2.
                   device=GPU_NAME)
        results.append(run)
        done.add(identity(run))
        # Append rather than rewrite, so saving stays cheap as the file grows.
        with PARTIAL.open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(run) + "\n")
        new_runs += 1
        if new_runs % SAVE_EVERY == 0:
            save_partial()
        bar.update(1)
    bar.close()

    del trainer, hypernet, target
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    save_partial()

began = time.perf_counter()
with Dashboard(f"Plasticity probe, order {ORDER}") as board:
    for seed in SEEDS:
        grid_at(seed, 0, restore_source=False,
                repetitions=REPETITIONS_AT_ZERO, board=board)
        grid_at(seed, 8, restore_source=True,
                repetitions=REPETITIONS_LATER, board=board)
save_partial()

SESSION_MIN = (time.perf_counter() - began) / 60
# Total time across every session, from the per-run timings, so a resumed run
# does not report only the minutes of its last session.
ELAPSED_MIN = sum(r.get("minutes", 0.0) for r in results)
print(f"\n{len(results)} probe runs in total, {new_runs} of them in this session "
      f"({SESSION_MIN:.1f} minutes).")
print("The model was verified unchanged after every one.")
missing = [(s, p) for s in SEEDS for p in (0, 8)
           if not any(r["seed_label"] == s and r["point"] == p for r in results)]
if missing:
    print("Still to do, as (seed, probe point):", missing)
    print("Run this cell again to continue from here.")

## 6. Pick the best rate, then average

For each seed, probe point and repetition the grid keeps one run: the one with the lowest
final training loss. That is what "how well can this model memorise" means, so the rate is
chosen on the same number that is then reported. There is no held-out split, because this
measures fitting, not generalising. The same rule is applied at both probe points, so the
comparison between the points is fair, but the number should be described as a
best-achievable fit rather than a score.

Then the repetitions are averaged.

In [ ]:
import statistics

def selected(seed, point, endpoint, task, arm, repetition):
    """The best rate for one seed, probe point and repetition."""
    runs = [r for r in results if r["seed_label"] == seed and r["point"] == point
            and r["endpoint"] == endpoint and r["task"] == task
            and r["arm"] == arm and r["repetition"] == repetition]
    return best_of_grid(runs, endpoint=endpoint) if runs else None

def endpoint_values(seed, point):
    """One final training loss per repetition, after the rate is chosen."""
    reps = REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER
    out = []
    for repetition in range(reps):
        picked = selected(seed, point, "E1", RANDOM_LABEL_TASK, "full", repetition)
        if picked is not None:
            out.append(picked["final_train_loss"])
    return out

mean = lambda values: sum(values) / len(values)

summary = {}
for seed in SEEDS:
    for point in (0, 8):
        summary[(seed, "E1", "full", point)] = endpoint_values(seed, point)

# A seed counts as finished only when both of its probe points produced values.
SEEDS_DONE = [s for s in SEEDS
              if summary[(s, "E1", "full", 0)] and summary[(s, "E1", "full", 8)]]

WIDTH = 92
def rule(title):
    print(f"\n{title}\n" + "-" * WIDTH)

print("=" * WIDTH)
print(f"RANDOM-LABEL FITTING, {STUDY_ID}, order {ORDER}, seeds {SEEDS_DONE}".center(WIDTH))
print(f"{len(results)} runs | {ELAPSED_MIN:.1f} min | task {RANDOM_LABEL_TASK}, "
      f"labels from seed {RANDOM_LABEL_SEED}".center(WIDTH))
print("=" * WIDTH)
if SEEDS_DONE != SEEDS:
    print()
    print("Incomplete. These seeds have no finished pair of probe points yet:",
          [s for s in SEEDS if s not in SEEDS_DONE])
    print("Run the grid cell again to continue. The tables cover the finished seeds.")

rule("TABLE 1. Final training loss after choosing the best rate, one value per repetition")
print(f"{'seed':>5}{'point':>8}{'reps':>6}{'mean':>10}{'sd':>9}  values")
print("-" * WIDTH)
for (seed, _, _, point), values in summary.items():
    if not values:
        continue
    sd = statistics.stdev(values) if len(values) > 1 else float("nan")
    shown = " ".join(f"{v:.4f}" for v in values)
    print(f"{seed:>5}{point:>8}{len(values):>6}{mean(values):>10.4f}{sd:>9.4f}  {shown}")
print("-" * WIDTH)
print("This is training loss on 2,000 images whose labels were shuffled. LOWER means the")
print("model memorised more of them, so lower means more capacity to change, not less.")

## 7. The decision rule

The adapter study declares plasticity loss only when both hold:

1. the change from t = 0 points the way that means loss, and
2. the change is larger than the noise floor.

For this measurement loss is a **rise**, because the number is a training loss: a model
that can no longer memorise ends higher. A fall is the opposite of plasticity loss.

The floor comes from the spread of the repetitions at t = 0:

    d = 1.96 * s * sqrt(1/n_0 + 1/n_8)

One check comes first. If the model never memorised anything, the loss sits near chance,
and then the number measures the step budget rather than the model. Chance for ten classes
is ln(10) = 2.303. The rule is only applied when both points are well below it.

In [ ]:
import math

CHANCE_LOSS = math.log(CLASSES)

# The random-label probe measures nothing until it has actually memorised some
# labels. A final loss sitting at chance means the budget was too small, not
# that capacity is unchanged, so the decision rule must not be applied to it.
# The study requires the probe to fit "well below chance loss".
def e1_worked(seed):
    values = [summary[(seed, "E1", "full", point)] for point in (0, 8)]
    return all(v and mean(v) < CHANCE_LOSS - 0.20 for v in values)

verdicts = {}

rule("TABLE 2. The decision rule, one row per seed")
print(f"{'seed':>5}{'t=0':>9}{'t=8':>9}{'change':>10}{'floor':>9}{'exceeds':>9}  verdict")
print("-" * WIDTH)
for seed in SEEDS_DONE:
    zero = summary[(seed, "E1", "full", 0)]
    later = summary[(seed, "E1", "full", 8)]
    if len(zero) < 2 or len(later) < 2:
        print(f"{seed:>5}  not enough repetitions to decide")
        continue
    mean_zero, mean_later = mean(zero), mean(later)
    change = mean_later - mean_zero
    floor = noise_floor(statistics.stdev(zero), statistics.stdev(zero),
                        len(zero), len(later))
    exceeds = abs(change) > floor
    if not e1_worked(seed):
        verdict = "VOID, the probe never memorised"
    else:
        verdict = ("PLASTICITY LOSS" if exceeds and change > 0 else
                   "no loss, more capacity to change" if exceeds else
                   "NO FINDING, inside the noise floor")
    verdicts[f"seed{seed}|E1"] = {
        "seed": seed, "t0": mean_zero, "t8": mean_later, "change": change,
        "probe_valid": e1_worked(seed), "noise_floor": floor,
        "exceeds_floor": exceeds, "direction_is_loss": change > 0,
        "verdict": verdict}
    print(f"{seed:>5}{mean_zero:>9.4f}{mean_later:>9.4f}{change:>+10.4f}"
          f"{floor:>9.4f}{('yes' if exceeds else 'no'):>9}  {verdict}")
print("-" * WIDTH)
print(f"Chance loss for {CLASSES} classes is ln({CLASSES}) = {CHANCE_LOSS:.3f}. Every mean above")
print("should sit well below it, or the run measured the step budget and not the model.")
for seed in SEEDS_DONE:
    if not e1_worked(seed):
        print(f"  WARNING, seed {seed}: the probe ended near chance. Raise "
              f"RANDOM_LABEL_STEPS or lower RANDOM_LABEL_IMAGES and rerun.")

rule("TABLE 3. Across seeds. This is the headline, not any single seed above")
across = {}
changes = [verdicts[f"seed{s}|E1"]["change"] for s in SEEDS_DONE
           if f"seed{s}|E1" in verdicts]
if changes:
    sd = statistics.stdev(changes) if len(changes) > 1 else float("nan")
    across["E1"] = {"seeds": len(changes), "mean_change": mean(changes),
                    "sd": sd, "changes": changes}
    shown = " ".join(f"{c:+.4f}" for c in changes)
    print(f"{len(changes)} seeds, mean change {mean(changes):+.4f}, sd {sd:.4f}")
    print(f"per seed: {shown}")
print("-" * WIDTH)

rule("How this compares with the October run on paired_generalization_v2")
# The October numbers, recomputed from its 189 saved random-label runs rather
# than copied from a write-up. Same method, different training run, and 5
# repetitions at t = 0 against only 2 at t = 8.
OCTOBER = {0: (1.3667, 0.6710, -0.6957, 0.2900),
           1: (1.5435, 0.7829, -0.7606, 0.4225),
           2: (1.3097, 0.6653, -0.6443, 0.1845)}
print(f"{'seed':>5}{'Oct change':>13}{'Oct floor':>12}{'here change':>14}{'here floor':>13}"
      f"  same direction")
print("-" * WIDTH)
for seed in SEEDS_DONE:
    here = verdicts.get(f"seed{seed}|E1")
    if here is None or seed not in OCTOBER:
        continue
    _, _, old_change, old_floor = OCTOBER[seed]
    agree = "yes" if (old_change < 0) == (here["change"] < 0) else "NO"
    print(f"{seed:>5}{old_change:>+13.4f}{old_floor:>12.4f}"
          f"{here['change']:>+14.4f}{here['noise_floor']:>13.4f}{agree:>16}")
print("-" * WIDTH)
print("The two runs trained different models from the same study design, so the sizes are")
print("not expected to match. What matters is the direction and whether it clears the floor.")
print("If a seed disagrees, the first thing to check is what differs between the two")
print("training runs, not whether the October finding has been overturned.")

## 8. Save

In [ ]:
import csv, datetime, platform

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
seed_label = "seeds" + "".join(str(s) for s in SEEDS_DONE)
EXPORT = CACHE / "diagnostics" / f"matched_random_labels_{ORDER}_{seed_label}_{stamp}"
EXPORT.mkdir(parents=True, exist_ok=True)

record = {
    "notebook": "41_matched_random_labels",
    "protocol": ("Fork and discard, as in reference/adapter-plasticity-study. The model is "
                 "snapshotted, a fresh task code is trained on an unseen task, the result "
                 "is measured, and the model is restored and verified unchanged."),
    "limitations": [
        "Two probe points only, t = 0 and t = 8. Nothing measures what happens "
        "between them for this endpoint.",
        "The rate is chosen on the same final training loss that is then reported. "
        "There is no held-out split, because the measurement is how well the model "
        "can memorise. The same rule runs at both points.",
        "The full arm only. The code-only and generator-only arms were settled on "
        "21 October and are not repeated here.",
        "Three seeds, two probe points each. The spread between seeds is wider "
        "than the mean change, so this rules out a large change rather than "
        "showing the change is zero.",
    ],
    "code_revision": HEAD, "study_id": STUDY_ID, "order": ORDER,
    "seeds": SEEDS, "seeds_complete": SEEDS_DONE,
    "probe_tasks": PROBE_TASKS, "random_label_task": RANDOM_LABEL_TASK,
    "rates": RATES, "steps": STEPS, "random_label_steps": RANDOM_LABEL_STEPS,
    "batch_size": BATCH_SIZE, "repetitions": {"0": REPETITIONS_AT_ZERO, "8": REPETITIONS_LATER},
    "elapsed_min": ELAPSED_MIN,
    "environment": {"device": DEVICE,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
                    "torch": torch.__version__, "python": sys.version.split()[0],
                    "platform": platform.platform()},
    "verdicts": verdicts,
    "arms": [a for a, _, _ in ARMS],
    "summary": {f"seed{s}|{e}|{a}|t{p}": v for (s, e, a, p), v in summary.items()},
    "across_seeds": across,
    "runs": results,
}
(EXPORT / "probe.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")

with (EXPORT / "runs.csv").open("w", newline="", encoding="utf-8") as stream:
    columns = ["seed_label", "point", "endpoint", "task", "learning_rate", "arm",
               "code_only", "repetition", "minutes",
               "steps", "final_train_loss", "final_train_accuracy", "test_accuracy",
               "test_loss", "effective_rank"]
    writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)

SAVE_VERIFIED = True
for name in ("probe.json", "runs.csv"):
    path = EXPORT / name
    try:
        text = path.read_text(encoding="utf-8")
        if name.endswith(".json"):
            json.loads(text)
        elif len(text.splitlines()) != len(results) + 1:
            raise ValueError("row count does not match the runs")
        print(f"  verified  {name:<14} {path.stat().st_size:>9,} bytes")
    except Exception as problem:
        SAVE_VERIFIED = False
        print(f"  FAILED    {name:<14} {problem}")

SAVED_LOCATION = str(EXPORT)
print("\nSaved locally:", EXPORT)
if SAVE_TO_R2 and SAVE_VERIFIED:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    for name in ("probe.json", "runs.csv"):
        store.upload(EXPORT / name, f"{destination}/{name}")
    SAVED_LOCATION = destination
    print("Uploaded to R2:", destination)
elif SAVE_TO_R2:
    print("Not uploading: the local files did not verify.")

## 9. Release the GPU

Only once every file has been written and read back. If anything failed to save the
runtime stays connected, so hours of probing are not lost.

In [ ]:
if not SAVE_VERIFIED:
    print("NOT releasing the GPU: some output files are missing or unreadable.")
    print("Rerun section 8 before closing this runtime, or the probe runs are lost.")
elif SEEDS_DONE != SEEDS:
    print("Results so far saved to:", SAVED_LOCATION)
    print("NOT releasing the GPU. These seeds are not finished yet:",
          [s for s in SEEDS if s not in SEEDS_DONE])
    print("Run the grid cell again to continue from where it stopped.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results saved to:", SAVED_LOCATION)
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
    print("Not running in Colab, so there is no runtime to release.")
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Every seed is finished. Share that path.")
    import gc
    for name in ("tasks", "random_label_set"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## 10. What this leaves

If all three seeds fall and clear their floors, random-label fitting is measured on the same
models as the five indicators, the matched probe and the random-direction control. The row
in the 3 November table stops carrying its "different training run" caveat, and the paper
can say that of everything measured here, one indicator agrees with the direct probe and
three point the other way, all on one set of models.

Three things it still does not settle.

| Question | Where it stands |
| --- | --- |
| Is the rate choice fair? | The rate is picked on the same loss that is reported. The same rule runs at both points, so the comparison holds, but the number is a best-achievable fit |
| What happens between t = 0 and t = 8? | Two points only. Notebook 40 measured five points for the direct probe; nothing similar exists for this one |
| Three seeds enough? | No. The spread between seeds is wide, so this rules out a large change rather than showing the change is zero |